# 업체 마스터 · 업체명 연결 → `clean_company` / `clean_company_name_link`

사업자등록번호가 있는 출처(계약정보 `raw_dapa_contract`, 입찰결과 `raw_dapa_bid_result`)로 업체 마스터를 만들고,
사업자번호가 없는 출처(국산화개발품목 계약업체, 방산업체 지정현황)의 업체명을 거기에 연결한다.

- 연결 키는 **업체명뿐**이다. 사업자번호끼리 맞추는 것이 아니라 이름을 맞추는 것이므로, 동명이인 위험이 있다.
- `match_type`: `exact`(사업자번호 1개) / `multi`(같은 이름에 사업자번호 여러 개) / `none`(못 찾음).
- **`multi`는 화면에서 쓰지 않는다.** 어느 법인인지 특정되지 않았다는 뜻이다.
- 정규화는 `notebooks/clean_b2_a7.ipynb`의 `norm_name`과 같은 규칙에 **영문 약자 한글 읽기**(LIG→엘아이지)를 더했다.


In [ ]:
import sys, re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user

NB_TAG = "notebooks/clean_company.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
def q(sql):
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
print("connected as", USER, "| pandas", pd.__version__)

## 1. 정규화 규칙

`norm_name`은 `clean_b2_a7.ipynb`와 **같은 규칙**이다(법인격 표기 제거 + 공백 정리). `clean_company.name_norm`에 그대로 넣는다.

`match_key`는 비교 전용 키다. `name_norm`에서 공백을 없애고 대문자로 맞춘다.
`match_key_rom`은 영문 약자를 한글 읽기로 펼친 키다. 방산업체 지정현황이 `LIG넥스원`으로 쓰고 계약정보가 `엘아이지넥스원`으로 쓰는 경우를 잡는다.
`에스엔티`→`에스앤티`는 표기 관행이라 한 줄로 보정한다. **비교에만 쓰고 저장하지 않는다.**

In [ ]:
_legal = re.compile(r"\(주\)|㈜|주식회사|\(유\)|유한회사|\(재\)|재단법인|\(사\)|사단법인")
def norm_name(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    t = _legal.sub(" ", str(s))
    t = re.sub(r"\s+", " ", t).strip()
    return t or None

def match_key(s):
    n = norm_name(s)
    return None if n is None else (n.replace(" ", "").upper() or None)

_ROM = {"A":"에이","B":"비","C":"씨","D":"디","E":"이","F":"에프","G":"지","H":"에이치","I":"아이",
        "J":"제이","K":"케이","L":"엘","M":"엠","N":"엔","O":"오","P":"피","Q":"큐","R":"알",
        "S":"에스","T":"티","U":"유","V":"브이","W":"더블유","X":"엑스","Y":"와이","Z":"지"}
def match_key_rom(s):
    k = match_key(s)
    if k is None:
        return None
    t = "".join(_ROM.get(ch, ch) if "A" <= ch <= "Z" else ch for ch in k)
    return t.replace("에스엔티", "에스앤티") or None

for s in ["엘아이지넥스원주식회사", "LIG넥스원", "(주)한화시스템"]:
    print(repr(s), "→", repr(norm_name(s)), "|", repr(match_key(s)), "|", repr(match_key_rom(s)))

## 2. 업체 마스터 `clean_company`

사업자번호가 있는 두 출처를 합친다. 같은 사업자번호가 여러 이름으로 나오면 **가장 많이 쓰인 원문**을 `name_raw`로 삼는다(스키마 주석 규칙).
`first_seen_source`는 계약정보에 먼저 나오면 `contract`, 입찰결과에만 있으면 `bid_result`.

In [ ]:
ct = q("SELECT vendor_biz_reg_no AS biz_reg_no, vendor_name AS name_raw, vendor_address AS address "
       "FROM raw_dapa_contract WHERE vendor_biz_reg_no IS NOT NULL AND vendor_biz_reg_no <> ''")
br = q("SELECT winner_biz_reg_no AS biz_reg_no, winner_name AS name_raw, winner_address AS address "
       "FROM raw_dapa_bid_result WHERE winner_biz_reg_no IS NOT NULL AND winner_biz_reg_no <> ''")
ct["src"], br["src"] = "contract", "bid_result"
src = pd.concat([ct, br], ignore_index=True)
print("행", len(src), "| 고유 사업자번호", src.biz_reg_no.nunique(),
      "| 계약", src.src.eq('contract').sum(), "| 입찰결과", src.src.eq('bid_result').sum())

# 사업자번호별 대표 이름 = 최빈 원문, 대표 주소 = 그 이름으로 가장 최근 관측(최빈)
def pick_mode(s):
    m = s.dropna()
    return m.mode().iat[0] if len(m) else None

comp = (src.groupby("biz_reg_no")
           .agg(name_raw=("name_raw", pick_mode),
                address=("address", pick_mode),
                has_contract=("src", lambda s: (s == "contract").any()))
           .reset_index())
comp["name_norm"] = comp.name_raw.map(norm_name)
comp["first_seen_source"] = comp.has_contract.map({True: "contract", False: "bid_result"})
print("업체 마스터", len(comp), "| name_norm NULL", int(comp.name_norm.isna().sum()))
comp.head()

### 시도 코드 — 주소 첫 토큰을 `ref_sido_map`에 맞춘다

`clean_dapa_contract.sido_code`와 같은 규칙이다. 못 맞추면 NULL로 둔다(추측하지 않는다).

In [ ]:
sido = q("SELECT token, sido_code FROM ref_sido_map")
smap = dict(zip(sido.token, sido.sido_code))
def first_token(a):
    if a is None or (isinstance(a, float) and pd.isna(a)):
        return None
    return str(a).strip().split(" ")[0] or None
comp["sido_code"] = comp.address.map(lambda a: smap.get(first_token(a)))
print("시도 매칭", int(comp.sido_code.notna().sum()), "/", len(comp),
      f"({comp.sido_code.notna().mean()*100:.1f}%)")
print("미매칭 주소 첫 토큰 상위:", comp.loc[comp.sido_code.isna(), "address"].map(first_token).value_counts().head(10).to_dict())

## 3. 업체명 연결 `clean_company_name_link`

이름 하나가 사업자번호 여러 개에 걸리면 `multi`다. 먼저 `match_key`로 찾고, 없으면 `match_key_rom`으로 한 번 더 찾는다.
`note`에는 어느 키로 붙었는지 남긴다(`key` / `key_rom`).

In [ ]:
idx, idx_rom = {}, {}
for nm, bz in zip(src.name_raw, src.biz_reg_no):
    k = match_key(nm)
    if k:
        idx.setdefault(k, set()).add(bz)
    kr = match_key_rom(nm)
    if kr:
        idx_rom.setdefault(kr, set()).add(bz)

def link_rows(names, source):
    rows = []
    for n in names:
        hit, via = None, None
        k = match_key(n)
        if k and k in idx:
            hit, via = idx[k], "key"
        else:
            kr = match_key_rom(n)
            if kr and kr in idx_rom:
                hit, via = idx_rom[kr], "key_rom"
        if hit is None:
            rows.append((source, n, norm_name(n), None, "none", 0, None))
        elif len(hit) == 1:
            rows.append((source, n, norm_name(n), sorted(hit)[0], "exact", 1, via))
        else:
            rows.append((source, n, norm_name(n), None, "multi", len(hit), via))
    return pd.DataFrame(rows, columns=["source", "name_raw", "name_norm", "biz_reg_no",
                                       "match_type", "match_count", "note"])

loc_names = q("SELECT DISTINCT contractor_name AS n FROM clean_dapa_localized_item "
              "WHERE contractor_name IS NOT NULL AND contractor_name <> ''").n.tolist()
dc_names = q("SELECT DISTINCT company_name AS n FROM raw_dapa_defense_company "
             "WHERE company_name IS NOT NULL AND company_name <> ''").n.tolist()
link = pd.concat([link_rows(loc_names, "localized_item"),
                  link_rows(dc_names, "defense_company")], ignore_index=True)

rep = (link.groupby(["source", "match_type"]).size().unstack(fill_value=0))
rep["연결률_%"] = ((rep.get("exact", 0) + rep.get("multi", 0)) / rep.sum(axis=1) * 100).round(1)
print(rep)

### 연결된 업체가 실제로 얼마를 계약했는지 (연결이 쓸모 있는지 확인)

`exact`로 붙은 국산화 계약업체만 계약액을 합쳐 본다. **계약액은 지원금이 아니라 조달 대금**이다.

In [ ]:
ex = link[(link.source == "localized_item") & (link.match_type == "exact")][["name_raw", "biz_reg_no"]]
amt = q("SELECT vendor_biz_reg_no AS biz_reg_no, contract_no, "
        "       MAX(CAST(REPLACE(total_contract_amount, ',', '') AS DECIMAL(20,0))) AS amt "
        "FROM raw_dapa_contract WHERE total_contract_amount IS NOT NULL "
        "GROUP BY vendor_biz_reg_no, contract_no")
per = amt.groupby("biz_reg_no").agg(contracts=("contract_no", "nunique"), amt=("amt", "sum")).reset_index()
chk = ex.merge(per, on="biz_reg_no", how="left")
chk["계약액_억"] = (pd.to_numeric(chk.amt, errors="coerce") / 1e8).round(0)
print("exact 연결 업체", len(chk), "| 계약 있는 업체", int(chk.contracts.notna().sum()))
chk.sort_values("계약액_억", ascending=False).head(15)[["name_raw", "contracts", "계약액_억"]]

## 4. 팀 서버 적재 (대상 테이블이 비어 있을 때만) + `meta_load_log`

`clean_company` → `clean_company_name_link` 순서로 넣는다(FK).
`match_type`이 `none`·`multi`면 `biz_reg_no`는 NULL이다(스키마 제약).

In [ ]:
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    return cur.fetchone()[0]

n_comp, n_link = count("clean_company"), count("clean_company_name_link")
print("적재 전 — clean_company", n_comp, "| clean_company_name_link", n_link)
assert n_comp == 0 and n_link == 0, "이미 값이 있다. db/reset_data.sql 로 비운 뒤 다시 실행한다."

comp_rows = [(r.biz_reg_no, r.name_norm, r.name_raw, r.address, r.sido_code, r.first_seen_source)
             for r in comp.itertuples()]
cur.executemany(
    "INSERT INTO clean_company (biz_reg_no, name_norm, name_raw, address, sido_code, first_seen_source) "
    "VALUES (%s, %s, %s, %s, %s, %s)", comp_rows)
print("clean_company", cur.rowcount)

link_rows_ = [(r.source, r.name_raw, r.name_norm, r.biz_reg_no, r.match_type, int(r.match_count), r.note)
              for r in link.itertuples()]
cur.executemany(
    "INSERT INTO clean_company_name_link (source, name_raw, name_norm, biz_reg_no, match_type, match_count, note) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s)", link_rows_)
print("clean_company_name_link", cur.rowcount)

log_stage(cur, "dapa_contract", "clean_company", "정제", "사업자번호 기준 업체 마스터(계약정보+입찰결과 합산)", len(comp_rows),
          "사업자번호 없는 행 제외. 같은 사업자번호의 여러 표기는 최빈 원문을 name_raw로 축약",
          f"{NB_TAG} §2 (pandas groupby)", USER)
for _src, _key in [("localized_item", "dapa_localized_item"), ("defense_company", "dapa_defense_company")]:
    _sub = link[link.source == _src]
    _linked = int((_sub.match_type != "none").sum())
    log_stage(cur, _key, "clean_company_name_link", "업체명 연결", f"{_src} 업체명 → clean_company", len(_sub),
              f"연결 {_linked}/{len(_sub)} (exact {int((_sub.match_type=='exact').sum())}, "
              f"multi {int((_sub.match_type=='multi').sum())}). multi는 법인 미특정이라 화면 사용 금지",
              f"{NB_TAG} §3 (업체명 키 일치)", USER)
conn.commit()
print("커밋 완료 —", count("clean_company"), count("clean_company_name_link"))

## 5. 적재 후 검증

In [ ]:
display(q("SELECT source, match_type, COUNT(*) n FROM clean_company_name_link GROUP BY source, match_type ORDER BY source, match_type"))
display(q("SELECT COUNT(*) companies, COUNT(sido_code) with_sido, COUNT(DISTINCT sido_code) sidos FROM clean_company"))
display(q("""
SELECT l.name_raw AS 업체, COUNT(DISTINCT i.part_mgmt_no) AS 국산화부품,
       COUNT(DISTINCT i.project_name) AS 무기체계,
       ROUND(SUM(t.amt)/100000000) AS 계약액_억
FROM clean_company_name_link l
JOIN clean_dapa_localized_item i ON i.contractor_name = l.name_raw
LEFT JOIN (SELECT vendor_biz_reg_no b, contract_no,
                  MAX(CAST(REPLACE(total_contract_amount, ',', '') AS DECIMAL(20,0))) amt
           FROM raw_dapa_contract GROUP BY vendor_biz_reg_no, contract_no) t ON t.b = l.biz_reg_no
WHERE l.source = 'localized_item' AND l.match_type = 'exact' AND i.fsc2 IN ('58','59')
GROUP BY l.name_raw ORDER BY 국산화부품 DESC LIMIT 15"""))

## 남은 일 (이 노트북이 하지 않은 것)

- **사명 변경 이력은 반영하지 않았다.** `두산인프라코어` → `에이치디현대인프라코어`, `한화디펜스` → `한화에어로스페이스` 같은 경우는 `none`으로 남는다. 옛 이름을 새 사업자번호에 잇는 것은 팀 판단이므로 별도 대응표가 필요하다.
- **`multi`는 화면에서 쓰지 않는다.** 어느 법인인지 특정되지 않았다.
- 국산화 계약업체 미연결의 대부분은 표기 문제가 아니라 **그 업체가 방사청 국내조달 계약정보에 없기 때문**이다(하청이거나 완제품 업체를 통해 납품). `LIKE` 확인 결과를 근거로 둔다.
- `clean_dapa_contract`가 비어 있어 계약액은 `raw_dapa_contract`에서 직접 집계했다. clean 계층이 채워지면 그쪽으로 바꾼다.
